# 🤖 Notebook 04 — Anomaly Detection Model

**Project:** Energy Fraud & Default Risk Detection  
**Layer:** ML (Gold → PCA → Isolation Forest → Risk Scores)  
**Author:** Zara Louise  
**Stack:** PySpark + Scikit-learn + MLflow + Unity Catalog  
**Depends:** `03_gold_features` → `energy_project.gold.features_distributor`

---

## 🎯 Goal

Train an unsupervised anomaly detection pipeline on the Gold feature matrix to generate a **risk score (0–100) per distributor** — identifying which distributors show abnormal consumption and default patterns.

---

## 🧠 Model Pipeline

**Step 1 — StandardScaler**  
Normalizes all 12 features to mean=0, std=1.  
Essential because features have very different scales:  
`avg_consumption_kwh` ranges from 3K to 5.6M while `avg_aging_24` ranges from 0 to 1.

**Step 2 — PCA**  
Reduces 12 features to the components that explain 95% of variance.  
Removes noise and multicollinearity before anomaly detection.  
Justified by the Big Data context — dimensionality reduction improves model robustness.

**Step 3 — Isolation Forest**  
Unsupervised anomaly detection — no labels required.  
Assigns an anomaly score to each distributor based on how easy it is to isolate from the rest.  
Distributors that are isolated quickly = anomalous = high risk.

**Step 4 — Risk Score (0–100)**  
Converts the raw Isolation Forest score to an interpretable 0–100 scale.  
Score 0 = normal behavior. Score 100 = maximum anomaly.

---

## 📐 Why This Approach

| Criteria | Justification |
|---|---|
| No labels available | Rules out all supervised models (Logistic Regression, etc.) |
| Fraud is rare by definition | Isolation Forest designed for imbalanced, rare-event detection |
| Multiple feature dimensions | PCA reduces noise before detection |
| Interpretable output needed | 0–100 score is business-friendly |
| Portfolio + MBA alignment | Combines Unsupervised ML (PCA) + Big Data (Spark + MLflow) |

---

## 📦 Outputs

| Table | Description |
|---|---|
| `energy_project.gold.risk_scores` | Risk score + anomaly flag per distributor |

**MLflow:** experiment, parameters, metrics and model registered in Databricks Model Registry.


In [0]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================
# PySpark — read Gold table and write risk scores
import pyspark.sql.functions as F

# NumPy and Pandas — bridge between Spark and sklearn
import numpy as np
import pandas as pd

# Scikit-learn — ML pipeline: scaling → PCA → Isolation Forest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline

# MLflow — experiment tracking and model registry
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

print("✅ All libraries imported — ready for anomaly detection!")

✅ All libraries imported — ready for anomaly detection!


In [0]:
# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================

CATALOG     = "energy_project"
SCHEMA_GOLD = "gold"

# Source table (Gold)
TABLE_G_FEATURES   = f"{CATALOG}.{SCHEMA_GOLD}.features_distributor"

# Destination table (Gold)
TABLE_G_RISK_SCORES = f"{CATALOG}.{SCHEMA_GOLD}.risk_scores"

# Model parameters
CONTAMINATION  = 0.1   # Expected fraction of anomalies (~10% of distributors)
PCA_VARIANCE   = 0.95  # Keep components that explain 95% of variance
RANDOM_STATE   = 42    # Reproducibility

# MLflow
MLFLOW_EXPERIMENT = "/Users/zarallouise@gmail.com/energy-fraud-detection"

# Feature columns (12 features — no identifiers)
FEATURE_COLS = [
    "avg_consumption_kwh",
    "std_consumption_kwh",
    "cv_consumption",
    "consumption_trend",
    "pct_residential",
    "pct_industrial",
    "pct_commercial",
    "avg_default_rate",
    "avg_aging_12",
    "avg_aging_24",
    "default_trend",
    "default_consumption_ratio",
]

print("📂 Gold source:")
print(f"   {TABLE_G_FEATURES}")
print()
print("🎯 Gold destination:")
print(f"   {TABLE_G_RISK_SCORES}")
print()
print(f"⚙️  Model parameters:")
print(f"   contamination  = {CONTAMINATION} ({int(CONTAMINATION*100)}% expected anomalies)")
print(f"   PCA variance   = {PCA_VARIANCE} (95% explained variance)")
print(f"   random_state   = {RANDOM_STATE}")
print()
print(f"🔬 Features: {len(FEATURE_COLS)}")
for f in FEATURE_COLS:
    print(f"   • {f}")

📂 Gold source:
   energy_project.gold.features_distributor

🎯 Gold destination:
   energy_project.gold.risk_scores

⚙️  Model parameters:
   contamination  = 0.1 (10% expected anomalies)
   PCA variance   = 0.95 (95% explained variance)
   random_state   = 42

🔬 Features: 12
   • avg_consumption_kwh
   • std_consumption_kwh
   • cv_consumption
   • consumption_trend
   • pct_residential
   • pct_industrial
   • pct_commercial
   • avg_default_rate
   • avg_aging_12
   • avg_aging_24
   • default_trend
   • default_consumption_ratio


In [0]:
# =============================================================================
# SECTION 1 — LOAD GOLD FEATURES
# =============================================================================
# Read the feature matrix from Gold and convert to Pandas.
# sklearn runs on a single node — toPandas() is safe here because
# we only have 105 rows (one per distributor).

df_gold = spark.table(TABLE_G_FEATURES)

# Separate identifiers from features
pdf = df_gold.toPandas()

# Identifier columns (not used in model)
id_cols = ["distributor_cnpj", "distributor_code", "distributor_name"]

# Feature matrix (only the 12 numeric features)
X = pdf[FEATURE_COLS].values

print(f"✅ Gold features loaded")
print(f"   Distributors: {len(pdf)}")
print(f"   Features:     {len(FEATURE_COLS)}")
print(f"   Matrix shape: {X.shape}")
print(f"\n🔍 Sample (first 3 rows, first 4 features):")
print(pdf[FEATURE_COLS[:4]].head(3).to_string())

✅ Gold features loaded
   Distributors: 105
   Features:     12
   Matrix shape: (105, 12)

🔍 Sample (first 3 rows, first 4 features):
   avg_consumption_kwh  std_consumption_kwh  cv_consumption  consumption_trend
0         25183.707804         5.452093e+04        2.164929       5.047942e+03
1         29164.402579         4.141342e+04        1.419999       1.091216e+04
2        884916.825811         3.177866e+06        3.591147       1.869765e+06


In [0]:
# =============================================================================
# SECTION 1.5 — BARTLETT'S TEST OF SPHERICITY
# =============================================================================
# Prerequisite check before applying PCA.
# Tests whether the correlation matrix is significantly different from
# the identity matrix — if not, PCA would have nothing meaningful to extract.
#
# H0: correlation matrix = identity matrix (variables are uncorrelated)
# H1: correlation matrix ≠ identity matrix (variables share structure)
#
# We need H1 to proceed with PCA.
# p-value < 0.05 → reject H0 → PCA is appropriate.

from scipy import stats

# Correlation matrix of the feature matrix
corr_matrix = np.corrcoef(X.T)
n, p        = X.shape

# Bartlett's test statistic
det         = np.linalg.det(corr_matrix)
chi2_stat   = -((n - 1) - (2 * p + 5) / 6) * np.log(det)
df          = p * (p - 1) / 2
p_value     = stats.chi2.sf(chi2_stat, df)

print("=" * 60)
print("📊 BARTLETT'S TEST OF SPHERICITY")
print("=" * 60)
print(f"\n   χ² statistic: {chi2_stat:.4f}")
print(f"   Degrees of freedom: {int(df)}")
print(f"   p-value: {p_value:.6f}")
print()
if p_value < 0.05:
    print("   ✅ p < 0.05 → Reject H0")
    print("   ✅ Correlation matrix ≠ identity matrix")
    print("   ✅ PCA is appropriate — proceeding with pipeline")
else:
    print("   ❌ p ≥ 0.05 → Fail to reject H0")
    print("   ❌ Variables may not be correlated enough for PCA")
print("=" * 60)

📊 BARTLETT'S TEST OF SPHERICITY

   χ² statistic: 1456.2174
   Degrees of freedom: 66
   p-value: 0.000000

   ✅ p < 0.05 → Reject H0
   ✅ Correlation matrix ≠ identity matrix
   ✅ PCA is appropriate — proceeding with pipeline


In [0]:
# =============================================================================
# SECTION 2 — ML PIPELINE: SCALER + PCA + ISOLATION FOREST
# =============================================================================
# Pipeline steps:
#   1. StandardScaler  → normalize features (mean=0, std=1)
#   2. PCA             → reduce dimensionality (keep 95% variance)
#   3. IsolationForest → assign anomaly score per distributor
#
# Everything tracked in MLflow: params, metrics, and model artifact.

mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(run_name="pca_isolation_forest_v1"):

    # ── Build pipeline ────────────────────────────────────────────────────────
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("pca",    PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)),
        ("model",  IsolationForest(
                       contamination=CONTAMINATION,
                       random_state=RANDOM_STATE,
                       n_estimators=100,
                   )),
    ])

    # ── Fit pipeline ──────────────────────────────────────────────────────────
    pipeline.fit(X)

    # ── Extract results ───────────────────────────────────────────────────────
    # raw score: more negative = more anomalous
    raw_scores    = pipeline.decision_function(X)
    # prediction: -1 = anomaly, 1 = normal
    predictions   = pipeline.predict(X)

    # ── Convert to 0–100 risk score ───────────────────────────────────────────
    # Invert and normalize: most anomalous → score 100, most normal → score 0
    min_s, max_s  = raw_scores.min(), raw_scores.max()
    risk_scores   = 100 * (1 - (raw_scores - min_s) / (max_s - min_s))

    # ── PCA info ──────────────────────────────────────────────────────────────
    pca_step       = pipeline.named_steps["pca"]
    n_components   = pca_step.n_components_
    explained_var  = pca_step.explained_variance_ratio_.sum()

    # ── Log to MLflow ─────────────────────────────────────────────────────────
    mlflow.log_params({
        "contamination":    CONTAMINATION,
        "pca_variance":     PCA_VARIANCE,
        "random_state":     RANDOM_STATE,
        "n_features_input": len(FEATURE_COLS),
        "n_components_pca": n_components,
        "n_estimators":     100,
    })
    mlflow.log_metrics({
        "pca_explained_variance": round(float(explained_var), 4),
        "n_anomalies_detected":   int((predictions == -1).sum()),
        "anomaly_rate":           round(float((predictions == -1).mean()), 4),
        "risk_score_mean":        round(float(risk_scores.mean()), 2),
        "risk_score_max":         round(float(risk_scores.max()), 2),
    })
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        signature=infer_signature(X, risk_scores),
        registered_model_name="energy-fraud-isolation-forest",
    )

    print(f"{'='*60}")
    print(f"🤖 MODEL RESULTS")
    print(f"{'='*60}")
    print(f"\n📐 PCA:")
    print(f"   Input features:    {len(FEATURE_COLS)}")
    print(f"   Components kept:   {n_components}")
    print(f"   Variance explained: {explained_var:.1%}")
    print(f"\n🚨 Anomalies detected: {(predictions == -1).sum()} / {len(predictions)}")
    print(f"\n📊 Risk score distribution:")
    print(f"   Min:  {risk_scores.min():.1f}")
    print(f"   Mean: {risk_scores.mean():.1f}")
    print(f"   Max:  {risk_scores.max():.1f}")
    print(f"\n✅ Model logged to MLflow!")
    print(f"{'='*60}")

2026/05/18 01:25:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f18d9cd0-1a58.cloud.databricks.com/ml/experiments/587872867295369/models/m-435410c001174c04a87ef2e1c5194502?o=7474659781825853
Registered model 'energy-fraud-isolation-forest' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.energy-fraud-isolation-forest': https://dbc-f18d9cd0-1a58.cloud.databricks.com/explore/data/models/workspace/default/energy-fraud-isolation-forest/version/2?o=7474659781825853


🤖 MODEL RESULTS

📐 PCA:
   Input features:    12
   Components kept:   7
   Variance explained: 95.0%

🚨 Anomalies detected: 11 / 105

📊 Risk score distribution:
   Min:  0.0
   Mean: 20.5
   Max:  100.0

✅ Model logged to MLflow!


In [0]:
# =============================================================================
# SECTION 3 — SAVE RISK SCORES TO GOLD
# =============================================================================
# Adds risk scores and anomaly flags back to the distributor identifiers
# and writes the final table to gold.risk_scores.

# Build result DataFrame
pdf_results = pdf[id_cols].copy()
pdf_results["risk_score"]    = risk_scores.round(2)
pdf_results["is_anomaly"]    = (predictions == -1)
pdf_results["anomaly_label"] = pdf_results["is_anomaly"].map(
    {True: "HIGH RISK", False: "NORMAL"}
)

# Sort by risk score descending (highest risk first)
pdf_results = pdf_results.sort_values("risk_score", ascending=False).reset_index(drop=True)
pdf_results["risk_rank"] = pdf_results.index + 1

# Convert back to Spark and write to Gold
df_risk_scores = spark.createDataFrame(pdf_results)

(
    df_risk_scores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_G_RISK_SCORES)
)

print(f"✅ gold.risk_scores written successfully!")
print(f"\n{'='*60}")
print(f"🚨 TOP 15 HIGHEST RISK DISTRIBUTORS")
print(f"{'='*60}")
print(pdf_results[["risk_rank", "distributor_code", "risk_score", "anomaly_label"]].head(15).to_string(index=False))
print(f"{'='*60}")

✅ gold.risk_scores written successfully!

🚨 TOP 15 HIGHEST RISK DISTRIBUTORS
 risk_rank distributor_code  risk_score anomaly_label
         1          ENEL CE      100.00     HIGH RISK
         2      ELETROPAULO       76.24     HIGH RISK
         3              CEA       75.63     HIGH RISK
         4          CEMIG-D       69.70     HIGH RISK
         5    EQUATORIAL MA       69.68     HIGH RISK
         6           CEGERO       69.40     HIGH RISK
         7       LIGHT SESA       67.34     HIGH RISK
         8          ENEL RJ       62.66     HIGH RISK
         9           CERSUL       59.57     HIGH RISK
        10    EQUATORIAL PA       58.38     HIGH RISK
        11            CERIS       56.56     HIGH RISK
        12           CEDRAP       48.52        NORMAL
        13        COPEL-DIS       45.26        NORMAL
        14    EQUATORIAL GO       39.83        NORMAL
        15            CEDRI       39.52        NORMAL


In [0]:
# =============================================================================
# SECTION 4 — FINAL VALIDATION
# =============================================================================

df_gold_scores = spark.table(TABLE_G_RISK_SCORES)

print("=" * 60)
print("🔍 VALIDATION — gold.risk_scores")
print("=" * 60)

print(f"\n📊 Total distributors scored: {df_gold_scores.count()}")

print(f"\n🚨 Anomaly summary:")
df_gold_scores.groupBy("anomaly_label").count().orderBy("anomaly_label").show()

print(f"\n📊 Risk score statistics:")
df_gold_scores.select("risk_score").describe().show()

print("=" * 60)
print("✅ MODEL COMPLETE — pipeline ready for Power BI dashboard!")
print("=" * 60)

🔍 VALIDATION — gold.risk_scores

📊 Total distributors scored: 105

🚨 Anomaly summary:
+-------------+-----+
|anomaly_label|count|
+-------------+-----+
|    HIGH RISK|   11|
|       NORMAL|   94|
+-------------+-----+


📊 Risk score statistics:
+-------+------------------+
|summary|        risk_score|
+-------+------------------+
|  count|               105|
|   mean|20.487619047619038|
| stddev|20.164567483002376|
|    min|               0.0|
|    max|             100.0|
+-------+------------------+

✅ MODEL COMPLETE — pipeline ready for Power BI dashboard!


In [0]:
# =============================================================================
# SECTION 4.5 — SILHOUETTE SCORE
# =============================================================================
# Measures how well-separated anomalies are from normal distributors.
# Range: -1 to 1
#   > 0.5 → strong separation
#   0.2–0.5 → reasonable separation
#   < 0.2 → weak separation
#
# We use the PCA-transformed data (after scaling) as input,
# and the binary anomaly label (-1 / 1) as cluster assignment.

from sklearn.metrics import silhouette_score

# Get the PCA-transformed representation (after scaler + PCA, before model)
X_transformed = pipeline[:-1].transform(X)  # scaler + PCA only

sil_score = silhouette_score(X_transformed, predictions)

print("=" * 60)
print("📊 SILHOUETTE SCORE")
print("=" * 60)
print(f"\n   Score: {sil_score:.4f}")
print()
if sil_score > 0.5:
    print("   ✅ Strong separation between anomalies and normal")
elif sil_score > 0.2:
    print("   ✅ Reasonable separation — model is meaningful")
else:
    print("   ⚠️  Weak separation — consider tuning contamination")
print()
print(f"   Interpretation: anomalous distributors are")
print(f"   {sil_score:.1%} separated from normal ones in PCA space")
print("=" * 60)

📊 SILHOUETTE SCORE

   Score: 0.5206

   ✅ Strong separation between anomalies and normal

   Interpretation: anomalous distributors are
   52.1% separated from normal ones in PCA space
